## Question Answering

In [ ]:
# virtual asssitants like Alexa, Siri, and Google Assistant

In [1]:
""" 
1. Extractive: Extract the answer from a given context.
2. Abstractive: Generate an answer from the context that correctly answers the question.
"""

' \n1. Extractive: Extract the answer from a given context.\n2. Abstractive: Generate an answer from the context that correctly answers the question.\n'

In [2]:
""" 
1. Finetune DistilBERT [https://huggingface.co/distilbert/distilbert-base-uncased] on the SQUAD dataset [https://huggingface.co/Squad/datasets] for extractive question answering.
2. Use your finetuned model for inference.

"""

' \n1. Finetune DistilBERT [https://huggingface.co/distilbert/distilbert-base-uncased] on the SQUAD dataset [https://huggingface.co/Squad/datasets] for extractive question answering.\n2. Use your finetuned model for inference.\n\n'

In [3]:
!pip install transformers datasets evaluate

In [4]:
from huggingface_hub import notebook_login

notebook_login()

## Load the SQUaD Dataset

In [6]:
from datasets import load_dataset

squad = load_dataset("squad", split = "train[:5000]")

In [10]:
squad = squad.train_test_split(test_size = .2)

In [11]:
squad.shape

{'train': (4000, 5), 'test': (1000, 5)}

In [12]:
squad["train"][0]

{'id': '56bfdbf2a10cfb1400551344',
 'title': 'Beyoncé',
 'context': 'Beyoncé has worked with Tommy Hilfiger for the fragrances True Star (singing a cover version of "Wishing on a Star") and True Star Gold; she also promoted Emporio Armani\'s Diamonds fragrance in 2007. Beyoncé launched her first official fragrance, Heat in 2010. The commercial, which featured the 1956 song "Fever", was shown after the water shed in the United Kingdom as it begins with an image of Beyoncé appearing to lie naked in a room. In February 2011, Beyoncé launched her second fragrance, Heat Rush. Beyoncé\'s third fragrance, Pulse, was launched in September 2011. In 2013, The Mrs. Carter Show Limited Edition version of Heat was released. The six editions of Heat are the world\'s best-selling celebrity fragrance line, with sales of over $400 million.',
 'question': 'How many editions of Heat have been launched?',
 'answers': {'text': ['six editions'], 'answer_start': [654]}}

In [13]:
# answers: starting location of the answer and the answer text
# context: background information from which the model needs to extract the answer.
# question: the question a model should answer

## Preprocess

In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [15]:
""" 
1. Some examples in a dataset may have a very long context that exceeds the maximum input length of the model. 
To deal with longer sequences, truncate only the context by setting truncation="only_second".

2. Next, map the start and end positions of the answer to the original context by setting return_offset_mapping=True.

3. With the mapping in hand, now you can find the start and end tokens of the answer. 
Use the sequence_ids method to find which part of the offset corresponds to the question and which corresponds to the context.
"""

' \n1. Some examples in a dataset may have a very long context that exceeds the maximum input length of the model. \nTo deal with longer sequences, truncate only the context by setting truncation="only_second".\n\n2. Next, map the start and end positions of the answer to the original context by setting return_offset_mapping=True.\n\n3. With the mapping in hand, now you can find the start and end tokens of the answer. \nUse the sequence_ids method to find which part of the offset corresponds to the question and which corresponds to the context.\n'

In [16]:
# create a function to truncate and map the start and end tokens of the answer to the context
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label it (0, 0)
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [17]:
tokenized_squad = squad.map(preprocess_function, batched=True, remove_columns=squad["train"].column_names)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [18]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

## Train

In [19]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

model = AutoModelForQuestionAnswering.from_pretrained("distilbert/distilbert-base-uncased")

W0405 13:34:22.749000 16923 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
training_args = TrainingArguments(
    output_dir="my_awesome_qa_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad["train"],
    eval_dataset=tokenized_squad["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

  0%|          | 0/750 [00:00<?, ?it/s]

/Users/koushalsmodi/Desktop/MachineLearning/MachineLearningProjects/NLP/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 2.30061674118042, 'eval_runtime': 29.7261, 'eval_samples_per_second': 33.64, 'eval_steps_per_second': 2.119, 'epoch': 1.0}
{'loss': 2.7579, 'grad_norm': 23.399152755737305, 'learning_rate': 6.666666666666667e-06, 'epoch': 2.0}


/Users/koushalsmodi/Desktop/MachineLearning/MachineLearningProjects/NLP/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 1.6914318799972534, 'eval_runtime': 30.7509, 'eval_samples_per_second': 32.519, 'eval_steps_per_second': 2.049, 'epoch': 2.0}


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 1.6381069421768188, 'eval_runtime': 32.696, 'eval_samples_per_second': 30.585, 'eval_steps_per_second': 1.927, 'epoch': 3.0}
{'train_runtime': 1143.971, 'train_samples_per_second': 10.49, 'train_steps_per_second': 0.656, 'train_loss': 2.3101871744791667, 'epoch': 3.0}


TrainOutput(global_step=750, training_loss=2.3101871744791667, metrics={'train_runtime': 1143.971, 'train_samples_per_second': 10.49, 'train_steps_per_second': 0.656, 'total_flos': 1175877900288000.0, 'train_loss': 2.3101871744791667, 'epoch': 3.0})

In [22]:
trainer.push_to_hub()

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/koushalsmodi/my_awesome_qa_model/commit/d97f069d887bd5d1f7d9747e154e1420c5eb718f', commit_message='End of training', commit_description='', oid='d97f069d887bd5d1f7d9747e154e1420c5eb718f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/koushalsmodi/my_awesome_qa_model', endpoint='https://huggingface.co', repo_type='model', repo_id='koushalsmodi/my_awesome_qa_model'), pr_revision=None, pr_num=None)

## Inference

In [23]:
question = "How many programming languages does BLOOM support?"
context = "BLOOM has 176 billion parameters and can generate text in 46 languages natural languages and 13 programming languages."

In [24]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("my_awesome_qa_model")
inputs = tokenizer(question, context, return_tensors="pt")

In [25]:
import torch
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained("my_awesome_qa_model")
with torch.no_grad():
    outputs = model(**inputs)

In [26]:
answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

In [27]:
predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
tokenizer.decode(predict_answer_tokens)

'46 languages natural languages and 13'